# Problem 4. Texture Patches with Five Percent of the Labels

**Day 2 · Problem 4 · 40 points · open-ended.**
**Compute:** `optional-colab-l4`. The CPU path is required and is what the grader runs; the `Accelerator extension` section at the end is optional.

**Allowed tools.** Python's standard library, NumPy, and PyTorch only, on a CPU. No web access, downloads, external datasets, model hubs, pretrained weights, or checkpoints you did not create in the same run.

## Task

Each input is a $10\times10$ grey patch with pixel values in $[0,1]$ showing one of three textures:

* class `0`, **stripes**: a sinusoidal grating at a random orientation, period, and phase;
* class `1`, **checker**: a product of two perpendicular gratings (a rotated checkerboard) at a random orientation, period, and phases;
* class `2`, **dots**: three to five blurred bright spots at random places on a darker background.

Contrast varies from patch to patch and every pixel carries strong additive noise before clipping to $[0,1]$, so single patches can be ambiguous.
Only **5%** of the training pool is labelled: 40 labelled patches against 760 unlabelled ones.

**Two scanners.** The 40 labelled training patches were recorded on an older *scanner A*.
The validation, unlabelled-pool, and test patches all come from the deployment *scanner B*, which records the same three textures with the same class definitions but adds its own systematic distortion; that distortion is not described here.
The unlabelled pool is your only large sample of scanner-B patches.
Build the most accurate classifier you can for scanner-B patches, using the unlabelled pool.

**Metric (higher is better).** Accuracy on the locked test patches (`r2_001_data.accuracy_per_example`).
Predictions are `int64` labels of shape `(N,)` with values in `{0, 1, 2}`.

## Data

| rows | count | how to get them | may enter training? | labels visible? | used for selection? |
|---|---:|---|---|---|---|
| labelled training | 40 | `r2_001_data.train_rows("texture")` | yes | yes | no |
| labelled validation | 40 | `r2_001_data.val_rows("texture")` | **no** | yes | **yes** |
| unlabelled pool | 760 | `r2_001_data.unlabelled_rows("texture")` | yes, without true labels (pseudo-labels, clustering, and so on) | **never** | no |
| locked test | 400 | only inside `final_test_score` | **no** | no | **no** |

Inputs are float32 `(N, 1, 10, 10)`; labels are `int64` `(N,)`.
`r2_001_data.simulate` may **not** be used for this problem (it would create labelled patches).
Any statistic you fit (a mean image, feature standardization, cluster centres, and so on) may use only the labelled training and unlabelled-pool rows: computed from validation patches it counts as training on them, and `predict` must label each patch on its own (a statistic computed across the batch of test patches it receives counts as fitting on test rows).

## Committed baseline

`r2_001_data.baseline_score("texture")` trains `r2_001_data.TextureCNN` (`Conv2d(1, 8, 3)`, ReLU, `Conv2d(8, 16, 3)`, ReLU, flatten, `Linear(576, 3)`) on the 40 labelled training patches only: `torch.manual_seed(20261101)`, then construct, Adam with learning rate $0.01$, $150$ full-batch steps of cross-entropy.
It scores **B = 0.4850** at the frozen seed.
With 40 validation patches each one is worth $0.025$ of validation accuracy, so validation separates large effects only.

## Fixed elements (all are graded)

* **Interface and seam.** Your approach exposes `fit(labelled_x, labelled_y, pool_x)` and `predict(x)`; `predict` returns the `(N,)` labels and is the `predict_fn` you pass to `final_test_score`. Every batch that reaches a model's `forward` during training passes through a seam that records `r2_001_data.split_of("texture", row)` for the **clean** rows (record before adding any input noise; recording the first training call of each model you fit is enough) into a set `seam`; assert that no recorded value is `"val"` or `"test"`. A feature pipeline complies the same way: if a model is trained on features computed from patches (hand-made features, embeddings, or any other transform), record `split_of` for the raw patches behind each training batch at the feature-extraction step, before the features reach the model.
* **Budget.** Create `budget = r2_001_data.StepBudget(1500)` before any fitting.
  **Every** optimizer you use, including any per-example refinement inside `predict`, is wrapped with `budget.wrap(...)`; at most 1,500 optimizer steps in total.
  An optimizer over a whole batch of examples costs one step per update, not one per example.
  Only non-iterative closed-form computation is free; any iterative solver must run as a wrapped optimizer so that every iteration counts.
  `budget.elapsed()`, measured after the test call, must be `< 60` seconds on the grading CPU.
* **Protocol.** Fit only on the rows this statement names, make every choice on `r2_001_data.val_rows("texture")`, and reach the locked test rows only through `r2_001_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task="texture")`, called exactly once, after all fitting and selection.
  Assert `r2_001_data.test_call_count("texture") == 0` immediately before that call and `== 1` after it, and print the returned `TestScore`.
  `r2_001_data.baseline_score("texture")` may be called (it is not counted) to print the baseline under the same protocol.
* **Seam values allowed:** `{"train", "unlabelled"}`.
* **Run clean.** The notebook runs top to bottom in a fresh kernel without errors; delete dead experiments.

## Grading

**Test score (24 points, 60%).** The grader re-executes this notebook top to bottom on a CPU under `SEED = 20261101` and reads the point estimate (`.score`) of your single `final_test_score` call.
The committed baseline scores **B = 0.4850** and the reference solution scores **R = 0.9900** at this seed; let `g = |B - R| = 0.5050`.

* **24 points** (100% of the score portion) if your accuracy is `>= 0.8638` (that is, `>= R - 0.25 g`);
* **14.4 points** (60%) if it is `> 0.4850` (that is, `> B`);
* **6 points** (25%) if it is `> 0.3588` (that is, `> B - 0.25 g`);
* **0 points** otherwise.

A protocol violation (a second `final_test_score` call, a seam value outside the allowed set, the step budget or time limit exceeded, a banned tool, or any use of validation or test rows other than their stated roles) scores **0** for the test-score portion.

**Writeup (16 points, 40%).** Four parts, 4 points each, graded against the rubric descriptors in `rubric.md`:

* **Approach:** the model or method, its inputs and features, and why it suits this problem.
* **Alternatives considered:** at least two, each with the validation evidence that ruled it out.
* **Evaluation:** the validation protocol, the single final test score with its 95% bootstrap interval and `n`, and the comparison with the baseline under the same protocol.
* **Limitations:** at least one failure mode and how you would test for it.

Every number in the writeup must be traceable to an output of this notebook.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()
_spec = importlib.util.spec_from_file_location("r2_001_data", DATA_DIR / "r2_001_data.py")
r2_001_data = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(r2_001_data)
assert r2_001_data.SEED == SEED

x_lab, y_lab = r2_001_data.train_rows("texture")
x_pool = r2_001_data.unlabelled_rows("texture")
x_val, y_val = r2_001_data.val_rows("texture")
print(x_lab.shape, x_pool.shape, x_val.shape, torch.bincount(y_lab).tolist())
print("baseline recipe:", r2_001_data.BASELINE_RECIPES["texture"])


## Your work

Baseline under the protocol, your semi-supervised method, the validation evidence (including at least one ablation), seam, budget, the single test call, and answer-check assertions.

In [ ]:
# Your approach: fit, predict, validation evidence, the single test call, and assertions.


## Writeup

**Approach.** *Replace.*

**Alternatives considered.** *Replace.*

**Evaluation.** *Replace.*

**Limitations.** *Replace.*

## Accelerator extension

*Optional; Colab L4 only; never executed by the grader and never required for credit. The graded score is always the CPU run above.*

On a Colab L4, run the same semi-supervised loop with a wider CNN, more rounds, and many augmented views of each pool patch per round (for example random flips and 90-degree rotations, which preserve every texture class), using mini-batches and `bfloat16` autocast:

```python
L4_CONFIG = {"device": "cuda", "amp_dtype": torch.bfloat16, "channels": (32, 64), "rounds": 6,
             "views_per_patch": 8, "steps_per_round": 2000, "batch_size": 1024, "lr": 3e-3}
```

**What to observe.** How validation accuracy and the per-class pseudo-label counts evolve over rounds, and whether more rounds help or start to amplify one class (confirmation bias).